# Multi-Dataset LoRA Fine-tuning Comparison
## Train Mistral-7B on 5 Different Datasets and Compare Results

This notebook:
1. ✅ Loads 5 different training datasets
2. ✅ Trains a separate LoRA model on each dataset
3. ✅ Evaluates all models on the same test set (for fair comparison)
4. ✅ Creates comprehensive visualizations to show which dataset works best

**Datasets:**
- prompt1-eng.json
- Prompt2-chinese.json (or Prompt2-eng.json - check your files)
- prompt3-chineese.json
- rewritten_prompt4-chineese.json
- rewritten_prompt5-chineese.json

**⚠️ IMPORTANT:** Set `DATA_DIR` in Cell 4 to point to your data files!  
Default: `/content/sample_data/`

## 1. Setup and Installation

In [ ]:
# Install packages langdetect
!pip install -q transformers accelerate torch datasets evaluate rouge-score nltk bert-score sacrebleu sentencepiece protobuf
!pip install -q peft bitsandbytes scipy langdetect
!pip install -q matplotlib seaborn pandas
print("✓ Packages installed")


In [ ]:
import json
import torch
import pandas as pd
import numpy as np
from typing import List, Dict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import gc

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    logging
)
logging.set_verbosity_error()

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

from evaluate import load
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import nltk
nltk.download('punkt', quiet=True)

import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
from langdetect import detect, LangDetectException

# ====================================================
# METRIC COMPUTATION CONTROL
# Set RECOMPUTE_METRICS = True to re-enable ROUGE/BLEU/BERTScore.
# When False: legacy metrics are SKIPPED; existing CSV/JSON
# files remain the source of truth for those columns.
# Only the new empathy metric is computed on every run.
# ====================================================
RECOMPUTE_METRICS = False

print(f"RECOMPUTE_METRICS = {RECOMPUTE_METRICS}")
if not RECOMPUTE_METRICS:
    print("Legacy metrics (ROUGE, BLEU, BERTScore) will NOT be recomputed.")
    print("Existing CSV/JSON outputs preserve those values.")


In [ ]:
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = getpass("HuggingFace token: ")
login(token=HF_TOKEN)
print("✓ Logged in")

## 2. Dataset Configuration

In [ ]:
import os

# IMPORTANT: Set your data directory here
DATA_DIR = '/content/sample_data/'

# Dataset configurations
DATASETS = {
    'Prompt1-Eng': {
        'file': 'prompt1-eng.json',
        'language': 'en',
        'description': 'English Prompt 1'
    },
    'Prompt2-Eng': {
        'file': 'Prompt2-chinese.json',  # Note: Check actual filename
        'language': 'en',
        'description': 'English Prompt 2'
    },
    'Prompt3-Eng': {
        'file': 'prompt3-chineese.json',
        'language': 'en',
        'description': 'English Prompt 3'
    },
    'Prompt4-Chinese': {
        'file': 'rewritten_prompt4-chineese.json',
        'language': 'zh',
        'description': 'Chinese Prompt 4 (Rewritten)'
    },
    'Prompt5-Chinese': {
        'file': 'rewritten_prompt5-chineese.json',
        'language': 'zh',
        'description': 'Chinese Prompt 5 (Rewritten)'
    }
}

# Test set (for evaluation - same for all models)
TEST_SET = 'PsyQA_example.json'
TEST_SAMPLES = 50

# Verify all files exist
print("="*80)
print("CHECKING DATA FILES")
print("="*80)
print(f"Data directory: {DATA_DIR}\n")

all_files_found = True
for name, config in DATASETS.items():
    filepath = os.path.join(DATA_DIR, config['file'])
    if os.path.exists(filepath):
        file_size = os.path.getsize(filepath) / 1024  # KB
        print(f"✓ {config['file']} ({file_size:.1f} KB)")
    else:
        print(f"✗ MISSING: {config['file']}")
        all_files_found = False

test_path = os.path.join(DATA_DIR, TEST_SET)
if os.path.exists(test_path):
    file_size = os.path.getsize(test_path) / 1024
    print(f"✓ {TEST_SET} ({file_size:.1f} KB)")
else:
    print(f"✗ MISSING: {TEST_SET}")
    all_files_found = False

if not all_files_found:
    print("\n⚠ Some files are missing! Please check your data directory.")
    print(f"\nActual files in {DATA_DIR}:")
    if os.path.exists(DATA_DIR):
        for f in sorted(os.listdir(DATA_DIR)):
            if f.endswith('.json'):
                print(f"  - {f}")
else:
    print(f"\n✓ All files found!")

print("="*80)

## 3. Data Loading Functions

In [ ]:
def load_training_dataset(file_path: str) -> List[Dict]:
    """
    Load a training dataset.
    Assumes format: [{"question": ..., "description": ..., "answers": [...]}]
    """
    # Prepend data directory
    full_path = os.path.join(DATA_DIR, file_path)
    
    try:
        with open(full_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        processed = []
        for item in data:
            # Handle different possible formats
            if 'answers' in item and item['answers']:
                answer = item['answers'][0].get('answer_text', '') if isinstance(item['answers'][0], dict) else item['answers'][0]
            elif 'answer' in item:
                answer = item['answer']
            else:
                continue
            
            if not answer:
                continue
            
            processed.append({
                'question': item.get('question', ''),
                'description': item.get('description', ''),
                'answer': answer
            })
        
        return processed
    except Exception as e:
        print(f"Error loading {full_path}: {e}")
        return []


def load_test_dataset(file_path: str, max_samples: int = 50) -> List[Dict]:
    """
    Load test dataset (PsyQA format)
    """
    # Prepend data directory
    full_path = os.path.join(DATA_DIR, file_path)
    
    with open(full_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if max_samples:
        data = data[:max_samples]
    
    processed = []
    for item in data:
        if not item.get('answers') or not item['answers'][0].get('answer_text'):
            continue
        
        processed.append({
            'question': item['question'],
            'description': item.get('description', ''),
            'answer': item['answers'][0]['answer_text'],
            'questionID': item['questionID']
        })
    
    return processed

print("✓ Data loading functions defined")

## 4. Format Functions

In [ ]:
def format_prompt(question: str, description: str, answer: str = None) -> str:
    """
    Format training example in Mistral-Instruct format
    """
    if description:
        prompt = f"""<s>[INST] 你是一位专业的心理健康顾问。

问题：{question}

详细描述：{description}

请提供专业、有帮助、共情的回答。 [/INST]"""
    else:
        prompt = f"""<s>[INST] 你是一位专业的心理健康顾问。

问题：{question}

请提供专业、有帮助、共情的回答。 [/INST]"""
    
    if answer:
        return f"{prompt} {answer}</s>"
    else:
        return prompt


def prepare_dataset_for_training(data: List[Dict], tokenizer) -> Dataset:
    """
    Prepare dataset for training
    """
    texts = []
    for item in data:
        text = format_prompt(item['question'], item['description'], item['answer'])
        texts.append({'text': text})
    
    dataset = Dataset.from_list(texts)
    
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=512
        )
    
    dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    
    # Add labels
    def add_labels(example):
        example['labels'] = example['input_ids'].copy()
        return example
    
    dataset = dataset.map(add_labels)
    
    return dataset

print("✓ Format functions defined")

## 5. Model Training Function

In [ ]:
def train_lora_model(dataset_name: str, train_data: List[Dict], tokenizer, output_dir: str):
    """
    Train a LoRA model on a specific dataset
    """
    print(f"\n{'='*80}")
    print(f"TRAINING MODEL ON: {dataset_name}")
    print(f"Training samples: {len(train_data)}")
    print(f"{'='*80}\n")
    
    # Load base model
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    print("Loading base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        token=HF_TOKEN,
        device_map="auto",
        trust_remote_code=True,
    )
    
    model = prepare_model_for_kbit_training(model)
    
    # Apply LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    print("✓ LoRA applied")
    
    # Prepare dataset
    print("Preparing dataset...")
    train_dataset = prepare_dataset_for_training(train_data, tokenizer)
    print(f"✓ Prepared {len(train_dataset)} samples")
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=1e-5,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        report_to="none",
    )
    
    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )
    
    # Train
    print("\nStarting training...")
    trainer.train()
    print("✓ Training complete")
    
    # Save
    model.save_pretrained(output_dir)
    print(f"✓ Model saved to {output_dir}")
    
    return model

print("✓ Training function defined")

## 6. Evaluation Functions

In [ ]:
def generate_response(model, tokenizer, question: str, description: str, max_tokens: int = 200) -> str:
    """
    Generate response from model
    """
    prompt = format_prompt(question, description)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "[/INST]" in full_text:
        response = full_text.split("[/INST]")[1].strip()
    else:
        response = full_text
    
    return response


def calculate_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """
    Calculate all evaluation metrics
    """
    # ROUGE-L
    rouge = load('rouge')
    rouge_results = rouge.compute(
        predictions=predictions,
        references=references,
        rouge_types=['rougeL']
    )
    rouge_l = rouge_results['rougeL'] * 100
    
    # BLEU-4
    bleu_scores = []
    smoothing = SmoothingFunction().method1
    for pred, ref in zip(predictions, references):
        if not pred.strip():
            bleu_scores.append(0.0)
            continue
        score = sentence_bleu(
            [list(ref)],
            list(pred),
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
        bleu_scores.append(score)
    bleu_4 = np.mean(bleu_scores) * 100
    
    # BERTScore
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p.strip()]
    if valid_pairs:
        valid_preds, valid_refs = zip(*valid_pairs)
        P, R, F1 = bert_score(
            list(valid_preds),
            list(valid_refs),
            lang='zh',
            verbose=False,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        bert_p = P.mean().item() * 100
        bert_r = R.mean().item() * 100
        bert_f1 = F1.mean().item() * 100
    else:
        bert_p = bert_r = bert_f1 = 0.0
    
    return {
        'ROUGE-L': rouge_l,
        'BLEU-4': bleu_4,
        'BERTScore-P': bert_p,
        'BERTScore-R': bert_r,
        'BERTScore-F1': bert_f1
    }


def evaluate_model(model, tokenizer, test_data: List[Dict], dataset_name: str) -> Dict:
    """
    Evaluate a model on test set
    """
    print(f"\nEvaluating {dataset_name}...")
    
    predictions = []
    references = []
    
    for item in tqdm(test_data, desc="Generating"):
        pred = generate_response(model, tokenizer, item['question'], item['description'])
        predictions.append(pred)
        references.append(item['answer'])
    
    # NOTE: Legacy metric functions remain but are not called.
    # Set RECOMPUTE_METRICS = True to re-enable them.
    if RECOMPUTE_METRICS:
        metrics = calculate_metrics(predictions, references)
    else:
        metrics = {
            'ROUGE-L': float('nan'), 'BLEU-4': float('nan'),
            'BERTScore-P': float('nan'), 'BERTScore-R': float('nan'),
            'BERTScore-F1': float('nan')
        }
        print(f"\nSkipping legacy metrics for {dataset_name} (RECOMPUTE_METRICS=False).")
    
    print(f"✓ {dataset_name} generation complete.")
    if RECOMPUTE_METRICS:
        for metric, value in metrics.items():
            if not (value != value):   # skip NaN
                print(f"  {metric}: {value:.2f}")
    
    return {
        'metrics': metrics,
        'predictions': predictions,
        'references': references
    }

print("✓ Evaluation functions defined")

## 7. Load Test Set (Common for All Models)

In [ ]:
# Load test set
print(f"Loading test set from {TEST_SET}...")
test_data = load_test_dataset(TEST_SET, TEST_SAMPLES)
print(f"✓ Loaded {len(test_data)} test samples\n")

print("Sample test item:")
print(f"Q: {test_data[0]['question'][:100]}...")
print(f"A: {test_data[0]['answer'][:100]}...")

## 8. Load Tokenizer (Shared)

In [ ]:
# Load tokenizer once (shared across all models)
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✓ Tokenizer loaded")

## 9. Main Training Loop - Train on All 5 Datasets

In [ ]:
# Store all results
all_results = {}

for dataset_name, dataset_config in DATASETS.items():
    print(f"\n\n{'#'*80}")
    print(f"# PROCESSING: {dataset_name}")
    print(f"# File: {dataset_config['file']}")
    print(f"{'#'*80}\n")
    
    # Load training data
    print(f"Loading {dataset_config['file']}...")
    train_data = load_training_dataset(dataset_config['file'])
    
    if not train_data:
        print(f"⚠ No data loaded for {dataset_name}, skipping...")
        continue
    
    print(f"✓ Loaded {len(train_data)} training samples")
    
    # Train model
    output_dir = f"./lora-{dataset_name.lower().replace(' ', '-')}"
    model = train_lora_model(dataset_name, train_data, tokenizer, output_dir)
    
    # Evaluate model
    results = evaluate_model(model, tokenizer, test_data, dataset_name)
    all_results[dataset_name] = results
    
    # Clean up GPU memory
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    print(f"\n✓ Completed {dataset_name}\n")

print("\n" + "="*80)
print("ALL TRAINING AND EVALUATION COMPLETE!")
print("="*80)

## 10. Compare Results

In [ ]:
# Create comparison dataframe
comparison_data = []

for dataset_name, results in all_results.items():
    row = {'Dataset': dataset_name}
    row.update(results['metrics'])
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# Sort by BERTScore F1 (primary metric)
comparison_df = comparison_df.sort_values('BERTScore-F1', ascending=False)

print("\n" + "="*100)
print("RESULTS COMPARISON - ALL DATASETS")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

# Save to CSV
comparison_df.to_csv('dataset_comparison_results.csv', index=False)
print("\n✓ Results saved to dataset_comparison_results.csv")

## 11. Comprehensive Visualizations

In [ ]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# 1. Bar chart comparison of all metrics
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Dataset Comparison: LoRA Fine-tuning Performance', fontsize=18, fontweight='bold', y=0.995)

metrics = ['ROUGE-L', 'BLEU-4', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1']
colors = sns.color_palette("husl", len(comparison_df))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    
    # Sort by this metric
    plot_df = comparison_df.sort_values(metric, ascending=True)
    
    bars = ax.barh(plot_df['Dataset'], plot_df[metric], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2,
                f'{width:.2f}',
                ha='left', va='center', fontweight='bold', fontsize=10, 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    ax.set_xlabel('Score', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

# Remove empty subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.savefig('dataset_comparison_bars.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Bar chart saved to dataset_comparison_bars.png")

In [ ]:
# 2. Grouped bar chart - all metrics side by side
fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(comparison_df))
width = 0.15

metrics_to_plot = ['ROUGE-L', 'BLEU-4', 'BERTScore-F1']
colors_grouped = ['#3498db', '#e74c3c', '#2ecc71']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors_grouped)):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, comparison_df[metric], width, 
                   label=metric, color=color, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}',
                ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_xlabel('Dataset', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Multi-Metric Comparison Across Datasets', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Dataset'], rotation=45, ha='right')
ax.legend(fontsize=12, loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_comparison_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Grouped bar chart saved to dataset_comparison_grouped.png")

In [ ]:
# 3. Radar chart comparison
from math import pi

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection='polar'))

categories = ['ROUGE-L', 'BLEU-4', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1']
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# Plot each dataset
colors_radar = sns.color_palette("husl", len(comparison_df))

for idx, row in comparison_df.iterrows():
    values = [row[cat] for cat in categories]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Dataset'], 
            color=colors_radar[idx], alpha=0.7)
    ax.fill(angles, values, alpha=0.15, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_title('Performance Comparison: Radar Chart', size=16, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True)

plt.tight_layout()
plt.savefig('dataset_comparison_radar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Radar chart saved to dataset_comparison_radar.png")

In [ ]:
# 4. Heatmap of all metrics
fig, ax = plt.subplots(figsize=(12, 8))

# Prepare data for heatmap
heatmap_data = comparison_df.set_index('Dataset')[metrics]

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.2f',
    cmap='YlGn',
    linewidths=0.5,
    cbar_kws={'label': 'Score'},
    ax=ax,
    vmin=0,
    vmax=100
)

ax.set_title('Performance Heatmap: All Datasets and Metrics', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Metrics', fontsize=13, fontweight='bold')
ax.set_ylabel('Datasets', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('dataset_comparison_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Heatmap saved to dataset_comparison_heatmap.png")

## 12. Best Dataset Summary

In [ ]:
# Find best dataset for each metric
print("\n" + "="*100)
print("BEST DATASET FOR EACH METRIC")
print("="*100)

for metric in metrics:
    best_idx = comparison_df[metric].idxmax()
    best_row = comparison_df.loc[best_idx]
    print(f"\n🏆 {metric}:")
    print(f"   Winner: {best_row['Dataset']}")
    print(f"   Score: {best_row[metric]:.2f}")

# Overall best (by BERTScore F1)
print("\n" + "="*100)
print("OVERALL BEST DATASET (by BERTScore-F1)")
print("="*100)
best_overall = comparison_df.iloc[0]
print(f"\n🥇 {best_overall['Dataset']}")
print(f"\nScores:")
for metric in metrics:
    print(f"  {metric}: {best_overall[metric]:.2f}")
print("\n" + "="*100)

## 13. Save Detailed Results

In [ ]:
# Save detailed results to JSON
detailed_results = {}

for dataset_name, results in all_results.items():
    detailed_results[dataset_name] = {
        'metrics': results['metrics'],
        'sample_predictions': [
            {
                'question': test_data[i]['question'],
                'prediction': results['predictions'][i],
                'reference': results['references'][i]
            }
            for i in range(min(5, len(test_data)))
        ]
    }

with open('dataset_comparison_detailed.json', 'w', encoding='utf-8') as f:
    json.dump(detailed_results, f, ensure_ascii=False, indent=2)

print("✓ Detailed results saved to dataset_comparison_detailed.json")

## Empathy Scoring (New Metric)

Computes `empathy_score` using `facebook/bart-large-mnli` (zero-shot classification).
- Labels each response as **"empathetic"** vs **"not empathetic"** using natural language inference.
- `empathy_score` = probability assigned to the "empathetic" label (0.0 – 1.0).
- **Only English responses** are scored (detected via `langdetect`).
- Non-English responses receive `NaN`.
- ROUGE / BLEU / BERTScore are **not** recomputed here.
- Existing metric columns in saved files are preserved.

In [ ]:
# ====================================================
# EMPATHY SCORING FUNCTION
# Model: facebook/bart-large-mnli (zero-shot classification)
# Labels responses as 'empathetic' vs 'not empathetic' using NLI
# ====================================================

def compute_empathy_scores(texts, batch_size=8):
    """
    Score a list of texts for empathy (0.0 – 1.0).

    Uses facebook/bart-large-mnli zero-shot classification to label each
    response as 'empathetic' or 'not empathetic'. The empathy_score is the
    probability assigned to the 'empathetic' label.

    Only English-detected texts are scored.
    Non-English and empty strings receive float('nan').

    Args:
        texts (list[str]): Generated responses to score.
        batch_size (int): Number of texts per inference batch.

    Returns:
        list[float]: Per-text empathy probability (or NaN).
    """
    from transformers import pipeline as hf_pipeline

    device = 0 if torch.cuda.is_available() else -1
    scores = [float('nan')] * len(texts)

    english_indices, english_texts = [], []
    for i, text in enumerate(texts):
        if not text or not str(text).strip():
            continue
        try:
            lang = detect(str(text))
        except LangDetectException:
            lang = 'unknown'
        if lang == 'en':
            english_indices.append(i)
            english_texts.append(str(text)[:512])

    if not english_texts:
        print("No English responses detected — all empathy_score values will be NaN.")
        return scores

    print(f"Loading zero-shot classifier on device={device} ...")
    empathy_clf = hf_pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=device,
    )

    print(f"Scoring {len(english_texts)} / {len(texts)} English responses ...")
    for i in range(0, len(english_texts), batch_size):
        batch = english_texts[i:i + batch_size]
        batch_indices = english_indices[i:i + batch_size]
        raw = empathy_clf(batch, candidate_labels=["empathetic", "not empathetic"])
        if isinstance(raw, dict):  # single item returned as dict
            raw = [raw]
        for idx, res in zip(batch_indices, raw):
            scores[idx] = res['scores'][res['labels'].index('empathetic')]

    del empathy_clf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return scores

print("✓ compute_empathy_scores() defined")

In [ ]:
# ====================================================
# COMPUTE EMPATHY SCORES AND SAVE UPDATED RESULTS
# ====================================================

PREDICTIONS_SAVE_PATH = 'dataset_comparison_predictions.csv'
SUMMARY_CSV           = 'dataset_comparison_results.csv'

# --- Collect predictions from current run ---
pred_rows = []
for _ds_name, _res in all_results.items():
    for _pred, _ref in zip(_res.get('predictions', []), _res.get('references', [])):
        pred_rows.append({'dataset': _ds_name, 'prediction': _pred, 'reference': _ref})

predictions_df = pd.DataFrame(pred_rows) if pred_rows else pd.DataFrame(
    columns=['dataset', 'prediction', 'reference'])

# --- Merge prior empathy scores if available ---
if os.path.exists(PREDICTIONS_SAVE_PATH):
    _prior = pd.read_csv(PREDICTIONS_SAVE_PATH)
    if 'empathy_score' in _prior.columns and not predictions_df.empty:
        print(f"Merging prior empathy scores from {PREDICTIONS_SAVE_PATH} ...")
        predictions_df = predictions_df.merge(
            _prior[['dataset', 'prediction', 'empathy_score']],
            on=['dataset', 'prediction'], how='left')
    elif predictions_df.empty and 'empathy_score' in _prior.columns:
        print(f"No new predictions generated. Loading all from {PREDICTIONS_SAVE_PATH} ...")
        predictions_df = _prior
    else:
        predictions_df['empathy_score'] = float('nan')
else:
    predictions_df['empathy_score'] = float('nan')

# --- Score rows missing an empathy score ---
_mask = predictions_df['empathy_score'].isna()
if _mask.any():
    _new_scores = compute_empathy_scores(
        predictions_df.loc[_mask, 'prediction'].tolist(), batch_size=16)
    predictions_df.loc[_mask, 'empathy_score'] = _new_scores
    print(f"Scored {_mask.sum()} new predictions.")
else:
    print("All predictions already have empathy scores — nothing new to compute.")

# --- Save per-response file ---
predictions_df.to_csv(PREDICTIONS_SAVE_PATH, index=False)
print(f"Saved {len(predictions_df)} rows → {PREDICTIONS_SAVE_PATH}")

# --- Update summary CSV (preserve old metric columns, add empathy_score) ---
_mean_empathy = (
    predictions_df.groupby('dataset')['empathy_score']
    .mean().rename('empathy_score'))

if os.path.exists(SUMMARY_CSV):
    _summary = pd.read_csv(SUMMARY_CSV)
else:
    _rows = [{'Dataset': ds, **res['metrics']} for ds, res in all_results.items()
             if 'metrics' in res]
    _summary = pd.DataFrame(_rows) if _rows else pd.DataFrame()

if not _summary.empty:
    _summary['empathy_score'] = _summary['Dataset'].map(_mean_empathy)
    _summary.to_csv(SUMMARY_CSV, index=False)
    print(f"Updated {SUMMARY_CSV} with empathy_score column.")
    print(_summary[['Dataset', 'empathy_score']].to_string(index=False))


## Summary

This notebook:
1. ✅ Trained 5 separate LoRA models (one per dataset)
2. ✅ Evaluated all models on the same test set for fair comparison
3. ✅ Generated comprehensive visualizations:
   - Individual metric bar charts
   - Grouped bar chart (multi-metric)
   - Radar chart (all datasets overlay)
   - Heatmap (all metrics and datasets)
4. ✅ Identified the best dataset for each metric
5. ✅ Saved all results to CSV and JSON

**Files Generated:**
- `dataset_comparison_results.csv` - Summary table
- `dataset_comparison_detailed.json` - Full results with sample predictions
- `dataset_comparison_bars.png` - Individual metric comparisons
- `dataset_comparison_grouped.png` - Multi-metric grouped bars
- `dataset_comparison_radar.png` - Radar chart
- `dataset_comparison_heatmap.png` - Performance heatmap

Use this analysis to select the best dataset for your fine-tuning task!